# brain_forecast — First Run on Nibi

Minimal end-to-end test + resource profiling so you can size the SLURM allocation.

**Order of this notebook matters.** GPU lock (cell 1) must run before anything imports TensorFlow-bearing libraries, per the Nibi developer guide. `brain_forecast` itself does not import TF, but we keep the guide's pattern in case the movie-decomposition package is imported in the same kernel later.

## 1. Environment + GPU lock
Follows the Nibi dev guide exactly. On a **GPU node** keep `torch.zeros(1).cuda()`. On the **login node** comment it out and set `CUDA_VISIBLE_DEVICES=''`.

In [ ]:
import os
os.environ.pop('SSL_CERT_FILE', None)
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CUDA_VISIBLE_DEVICES'] = '-1'      # TF (if ever imported) -> CPU
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'          # PyTorch -> GPU 0

import torch
torch.zeros(1).cuda()  # lock PyTorch onto the GPU FIRST  (comment out on login node)
print('CUDA available:', torch.cuda.is_available())
print('Device        :', torch.cuda.get_device_name(0))
print('VRAM total    : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory/1e9))

## 2. Install brain_forecast (once per container/overlay)
Skip if already installed. `--no-deps` because the container/def already has every dependency — this avoids pip touching the pinned numpy.

In [ ]:
# Adjust the path to wherever you unpacked the package.
BRAIN_FORECAST_DIR = '/home/tamires/projects/rpp-aevans-ab/tamires/brain_forecast'
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', BRAIN_FORECAST_DIR,
                '--no-deps', '--quiet'], check=False)
import brain_forecast; print('brain_forecast', brain_forecast.__version__)

## 3. Resource probe — helpers
Call `snapshot('label')` at any point. At the end we print peak RAM and peak VRAM, which are what you size `--mem` and the GPU MIG slice against.

In [ ]:
import time, gc, resource
_marks = []
def snapshot(label):
    gc.collect()
    ru = resource.getrusage(resource.RUSAGE_SELF)
    ram_gb = ru.ru_maxrss / (1024**2)        # ru_maxrss is KB on Linux -> GB
    if torch.cuda.is_available():
        vram_gb = torch.cuda.max_memory_allocated() / 1e9
        vram_now = torch.cuda.memory_allocated() / 1e9
    else:
        vram_gb = vram_now = 0.0
    _marks.append((label, time.time(), ram_gb, vram_gb, vram_now))
    print(f'[{label:24s}] peakRAM={ram_gb:6.2f}GB  peakVRAM={vram_gb:6.2f}GB  nowVRAM={vram_now:5.2f}GB')

snapshot('start')

## 4. Point at your data
One parquet/CSV, long format: reserved columns `sub`, `start` (seconds), `cohort` (movie). Then any number of feature/target columns. Build the typed role lists by column-name pattern — do not hand-list ~6000 columns.

In [ ]:
import pyarrow.parquet as pq

DATA_PATH = '/home/tamires/projects/rpp-aevans-ab/tamires/DecomposingMovies/outputs/forecast_table.parquet'

# Read ONLY the schema (no data) to discover columns cheaply.
schema_cols = pq.ParquetFile(DATA_PATH).schema.names
print('total columns:', len(schema_cols))

# ---- EDIT THESE PATTERNS to match your real column names -----------------
static            = [c for c in schema_cols if c in ('age', 'sex', 'handedness')]
static_categorical= [c for c in static if c in ('sex', 'handedness')]
known_dynamic     = [c for c in schema_cols if c.startswith('mov_')]          # stimulus
observed_dynamic  = [c for c in schema_cols if c.startswith(('umap', 'pca'))] # brain history
target_cols       = [c for c in schema_cols if c.startswith('UDFC_')]         # DFC connections
# --------------------------------------------------------------------------

print(f'static={len(static)}  known={len(known_dynamic)}  '
      f'observed={len(observed_dynamic)}  targets={len(target_cols)}')

## 5. First experiment — START SMALL
For the very first run: **few targets, few horizons, benchmarks only (no TFT yet)**. This confirms the data loads, CV splits cleanly, and gives a RAM baseline before you spend GPU time on the TFT.

In [ ]:
from brain_forecast.data import load_bundle
from brain_forecast.features import TabularAdapter, SequenceAdapter
from brain_forecast.cv import StratifiedSubjectOutCV
from brain_forecast.evaluation import run_experiment
from brain_forecast.reporting import aggregate_scores, plot_horizon_curves

TARGET_SUBSET = target_cols[:5]   # start with 5 targets, scale up later

bundle = load_bundle(
    path=DATA_PATH,
    target_cols=TARGET_SUBSET,
    task_type='regression',
    static=static, static_categorical=static_categorical,
    known_dynamic=known_dynamic,
    observed_dynamic=observed_dynamic,
)
snapshot('bundle loaded')
print('subjects:', bundle.n_subjects(), '| cohorts:', list(bundle.cohorts()))

In [ ]:
results = run_experiment(
    bundle=bundle,
    predictor_specs=[
        {'name': 'persistence'},
        {'name': 'moving_average', 'kwargs': {'k': 5}},
        {'name': 'ar', 'kwargs': {'p': 5}},
    ],
    horizons_min=[0, 5, 15],          # 3 horizons to start
    cv=StratifiedSubjectOutCV(k_default=5, loso_threshold=10),
    tabular_adapter=TabularAdapter(ops=['lag', 'rolling', 'target_history', 'hrf'],
                                   k_lag=3, rolling_window=10, target_history_lags=5),
    output_dir='/home/tamires/scratch/bf_first_run',
)
snapshot('benchmarks done')
aggregate_scores(results, 'r2_mean', by=['predictor', 'horizon_min'])

## 6. TFT smoke test (1 horizon, capped epochs)
Only after the benchmarks pass. This is the GPU-heavy part; keep it tiny first to get a VRAM number.

In [ ]:
torch.cuda.reset_peak_memory_stats()
results_tft = run_experiment(
    bundle=bundle,
    predictor_specs=[{'name': 'tft', 'kwargs': {
        'max_epochs': 3,            # tiny: just to measure, not to converge
        'hidden_size': 32,
        'attention_head_size': 2,
        'batch_size': 64,
        'device': 'cuda',
    }}],
    horizons_min=[0],               # ONE horizon for the smoke test
    cv=StratifiedSubjectOutCV(k_default=5, loso_threshold=10),
    sequence_adapter=SequenceAdapter(window_min=5.0),
    output_dir='/home/tamires/scratch/bf_first_run_tft',
)
snapshot('tft smoke done')
results_tft.head()

## 7. Resource summary — use this to size SLURM
`peakRAM` → set `--mem` to ~1.5× this. `peakVRAM` → confirm it fits the MIG slice (the `1g.10gb` slice is **10 GB**). The per-step timings let you extrapolate `--time` for the full target/horizon grid.

In [ ]:
import pandas as pd
rows = []
for i, (label, t, ram, vram, vnow) in enumerate(_marks):
    dt = 0.0 if i == 0 else t - _marks[i-1][1]
    rows.append({'stage': label, 'dt_sec': round(dt, 1),
                 'peakRAM_GB': round(ram, 2), 'peakVRAM_GB': round(vram, 2)})
prof = pd.DataFrame(rows)
print(prof.to_string(index=False))
print()
peak_ram  = prof['peakRAM_GB'].max()
peak_vram = prof['peakVRAM_GB'].max()
print(f'PEAK RAM : {peak_ram:.2f} GB   -> suggest --mem={int(max(8, peak_ram*1.5))}G')
print(f'PEAK VRAM: {peak_vram:.2f} GB  -> MIG 1g.10gb gives 10 GB; '
      f"{'OK' if peak_vram < 9 else 'TOO BIG, request a larger slice'}")

## 8. Extrapolate to the full grid
Edit the multipliers to your real plan (e.g. all targets, 7 horizons, 5 folds, 30 epochs).

In [ ]:
# Time of the 1-horizon / 3-epoch TFT smoke run:
tft_dt = next(r['dt_sec'] for r in rows if r['stage'] == 'tft smoke done')

N_HORIZONS_FULL = 7      # e.g. [0,5,10,15,30,45,60]
EPOCH_SCALE     = 30/3   # full epochs vs the 3 used here
TARGET_SCALE    = len(target_cols)/len(TARGET_SUBSET)  # all targets vs the 5 used

est_sec = tft_dt * N_HORIZONS_FULL * EPOCH_SCALE * TARGET_SCALE
print(f'Smoke TFT (1 horizon, 3 epochs, 5 targets): {tft_dt/60:.1f} min')
print(f'Rough full-grid TFT estimate              : {est_sec/3600:.1f} h')
print('Add ~20% headroom; round --time up to the next hour.')